# Text-to-SQL-to-NoSQL Full Pipeline Runner

This notebook runs the complete pipeline: Setup, Data Download, Fine-tuning, and Web App Hosting.

## ⚠️ IMPORTANT: Enable GPU First!
1. **Runtime** > **Change runtime type** > **T4 GPU** (or better).
2. Prepare your `text-to-sql.zip` file.

## 1. Setup Environment

**Option A: Google Drive (Recommended for large files)**
1. Upload `text-to-sql.zip` (or `text-to-SQL.zip`) to your Google Drive root folder.
2. Run the cell below to mount Drive and copy.

**Option B: Direct Upload**
1. Drag and drop the zip file into the Files sidebar.
2. Run the cell below.

In [ ]:
import torch
if not torch.cuda.is_available():
    print("\u26a0\ufe0f WARNING: GPU not found! Training will be painfully slow.")
    print("\U0001f4a1 ACTION: Go to Runtime > Change runtime type > T4 GPU")

import os
import shutil
import zipfile
from google.colab import drive

# File configuration
possible_zips = ['text-to-sql.zip', 'text-to-SQL.zip', 'Text-to-SQL.zip']
target_zip = 'text-to-sql.zip'
possible_dirs = ['text-to-sql', 'text-to-SQL', 'Text-to-SQL']

# 1. Interactive Clean/Setup
should_process_zip = True
existing_dir = None

for d in possible_dirs:
    if os.path.exists(d) and os.path.isdir(d):
        existing_dir = d
        break

if existing_dir:
    print(f"\n📁 Found existing project folder: {existing_dir}")
    try:
        ans = input("❓ Do you want to DELETE it and re-upload/unzip? (y/n) [default: y]: ").strip().lower()
    except:
        ans = 'y' # Auto-yes if non-interactive (rare)
        
    if ans == 'n':
        print(f"Keeping existing folder. Entering {existing_dir}...")
        %cd {existing_dir}
        should_process_zip = False
    else:
        print(f"Removing {existing_dir}...")
        shutil.rmtree(existing_dir)

# 2. Process Zip (if needed)
if should_process_zip:
    found_path = None
    
    # Check Local
    for fname in possible_zips:
        if os.path.exists(fname):
            if zipfile.is_zipfile(fname):
                print(f"Found valid local file: {fname}")
                found_path = fname
                break
            else:
                print(f"Found CORRUPT local file: {fname}. Deleting it...")
                os.remove(fname)

    # Check Drive
    if not found_path:
        print("Checking Google Drive...")
        try:
            drive.mount('/content/drive')
            for fname in possible_zips:
                d_path = f'/content/drive/MyDrive/{fname}'
                if os.path.exists(d_path):
                    if zipfile.is_zipfile(d_path):
                        print(f"Found valid file in Drive: {fname}")
                        found_path = d_path
                        break
        except Exception as e:
            print(f"Drive check skipped: {e}")

    # Unzip logic
    if found_path:
        if found_path.startswith('/content/drive'):
            print(f"Copying {found_path}...")
            shutil.copy(found_path, target_zip)
        elif found_path != target_zip:
            os.rename(found_path, target_zip)

        print(f"Unzipping {target_zip}...")
        !unzip -o -q {target_zip}

        # Enter directory
        final_dir = None
        for d in possible_dirs:
            if os.path.isdir(d):
                final_dir = d
                break
        
        if final_dir:
            print(f"Entering directory: {final_dir}")
            %cd {final_dir}
        else:
            # Fallback search
            dirs = [d for d in os.listdir('.') if os.path.isdir(d) and 'text-to-sql' in d.lower()]
            if dirs:
                 print(f"Entering directory: {dirs[0]}")
                 %cd {dirs[0]}
            else:
                 print("Error: Could not find extracted project folder.")
    else:
        print("\n❌ ERROR: No valid 'text-to-sql.zip' found!")

## 2. Install Dependencies

In [ ]:
!pip install -r requirements.txt
!pip install pyngrok

## 3. Download Data
Downloads Spider and checks BirdBench setup. 
*Note: Script skips download if data exists.*

In [ ]:
print("--- Checking Spider Dataset ---")
!python scripts/download_spider.py

print("\n--- Checking BirdBench Dataset ---")
!python scripts/download_bird.py

print("\n--- Generating Synthetic DDL/DML Data ---")
!python scripts/generate_ddl_data.py

## 4. Fine-Tuning
Runs combined fine-tuning on Spider (4245-5660) and BirdBench (4783-6376).
*Note: Skips automatically if checkpoint exists.*

In [ ]:
!python src/train.py

## 5. Launch Web App
Starts the Flask app and exposes it via ngrok.

In [ ]:
from pyngrok import ngrok
import time


# --- NGROK AUTHENTICATION (REQUIRED) ---
# 1. Sign up at https://dashboard.ngrok.com/signup
# 2. Get your token at https://dashboard.ngrok.com/get-started/your-authtoken
# 3. Paste it inside the quotes below:
NGROK_AUTH_TOKEN = "383FWm3gwRtmSUBPFpUKfTvazKh_2GmCACxR5y7eTBrmWC47C"
if NGROK_AUTH_TOKEN != "383FWm3gwRtmSUBPFpUKfTvazKh_2GmCACxR5y7eTBrmWC47C":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print("\u26a0\ufe0f WARNING: Ngrok Auth Token not set! The tunnel will fail.")

# Start Flask in background
get_ipython().system_raw('python app.py &')

print("Waiting for app to start...")
time.sleep(10)

# Open tunnel
try:
    public_url = ngrok.connect(5000).public_url
    print(f"\n✅ App is running at: {public_url}")
    print("Click the link above to test properly!")
except Exception as e:
    print(f"Ngrok error: {e}")

# Keep cell running to keep app alive
while True:
    time.sleep(60)